In [1]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
dataset_id = "knkarthick/samsum"

In [3]:
!pip install datasets

In [4]:
!pip install py7zr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 14.1 MB/s eta 0:00:00


In [5]:
from datasets import load_dataset

# Load dataset from the hub
dataset = load_dataset(dataset_id)

print(f"Train dataset size: {len(dataset['train'])}")
print(f"Test dataset size: {len(dataset['test'])}")

# Train dataset size: 14732
# Test dataset size: 819

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

Train dataset size: 14731
Test dataset size: 819


In [6]:
from random import randrange


sample = dataset['train'][randrange(len(dataset["train"]))]
print(f"dialogue: \n{sample['dialogue']}\n---------------")
print(f"summary: \n{sample['summary']}\n---------------")

dialogue: 
Shiloh: Wow, did you hear they cast Henry Cavill as Geralt?
Casey: Welp... I'm not feeling very positive about this casting
Shiloh: I haven't really seen him in any roles, I don't watch many films or tv series, haha.
Shiloh: But he doesn't really look like Geralt to me
Casey: I've just checked the showrunner's twitter and still don't know what to think
Casey: Feels like they only did that to have a big name playing the lead role lol
Casey: To attract female audience, maybe
Shiloh: I guess there's makeup and clothes... that may change a lot, maybe it'll actually work
Casey: Who knows... I'm worried they'll butcher it, I really loved the game
Shiloh: Me too, but I guess we've gotta wait until it's on Netflix
Casey: Or until we know more about the rest of the cast :p
Shiloh: Well, that too... Keeping my fingers crossed for Eva Green as Yen
---------------
summary: 
Showrunner has announced on Twitter that Henry Cavill will play Geralt in Netflix series. Casey and Shiloh are uns

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id="google/flan-t5-small"

# Load tokenizer of FLAN-t5-base
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [8]:
def preprocess(batch):
    # Ensure all dialogues are strings
    dialogues = [str(d) if not isinstance(d, str) else d for d in batch["dialogue"]]
    summaries = [str(s) if not isinstance(s, str) else s for s in batch["summary"]]

    # Tokenize inputs
    #Converts dialogue text → token IDs (input_ids) and attention masks.
    #truncation=True: cuts off text longer than model’s max length.
    #padding=False: does no padding, so sequences keep variable length.
    model_inputs = tokenizer(dialogues, truncation=True, padding=False)

    # Tokenize targets
    labels = tokenizer(summaries, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [9]:
#Makes sure there are no None or weird datatypes (like numbers).
#Finds the longest dialogue and longest summary in your dataset.
#Useful to decide a max_length for padding in training.

all_input_ids = []
all_labels = []

for split_name in tokenized_dataset.keys():
    all_input_ids.extend(tokenized_dataset[split_name]["input_ids"])
    all_labels.extend(tokenized_dataset[split_name]["labels"])

max_source_length = max(len(ids) for ids in all_input_ids)
max_target_length = max(len(ids) for ids in all_labels)

print(f"Max source length: {max_source_length}")
print(f"Max target length: {max_target_length}")

Max source length: 512
Max target length: 95


In [10]:
def preprocess_function(sample, padding="max_length"):
    # coerce None to empty string
    #For T5 models, you need task prefixes (e.g. "summarize:").
    # Ensures that if dialogue is None, it becomes an empty string "".

    inputs = ["summarize: " + (item if isinstance(item, str) else "") for item in sample["dialogue"]]

    # tokenize inputs
    #Tokenizes the prefixed dialogues.
    #max_length=max_source_length: truncates if too long.
    # padding=max_length: pads all to the same fixed length (max_source_length).
    model_inputs = tokenizer(inputs, max_length=max_source_length, padding=padding, truncation=True)

    # tokenize targets
    targets = [(t if isinstance(t, str) else "") for t in sample["summary"]]
    labels = tokenizer(text_target=targets, max_length=max_target_length, padding=padding, truncation=True)

    # replace pad token id with -100 for loss masking
    #In loss calculation, we don’t want to consider padded tokens.
    if padding == "max_length":
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
        ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["dialogue", "summary", "id"]
)
print(f"Keys of tokenized dataset: {list(tokenized_dataset['train'].features)}")


Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Keys of tokenized dataset: ['input_ids', 'attention_mask', 'labels']


In [11]:
from transformers import AutoModelForSeq2SeqLM

# huggingface hub model id
model_id="google/flan-t5-small"

# load model from the hub
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [13]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=e54450e488cbcefc2374196cdd24c2bc496c42a1a28cead8b984a8af265d47d0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [14]:
import evaluate
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize
nltk.download("punkt")
nltk.download("punkt_tab") # Download the missing 'punkt_tab' resource

# Metric
metric = evaluate.load("rouge")

# helper function to postprocess text
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # rougeLSum expects newline after each sentence
    preds = ["\n".join(sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(sent_tokenize(label)) for label in labels]

    return preds, labels
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Fix predictions tuple issue
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert logits to token ids if needed
    predictions = np.argmax(predictions, axis=-1)

    # Replace -100 in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    return {}

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [15]:
from transformers import DataCollatorForSeq2Seq

# we want to ignore tokenizer pad token in the loss
label_pad_token_id = -100
# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=label_pad_token_id,
    pad_to_multiple_of=8
)


In [24]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Hugging Face repository id
repository_id = f"{model_id.split('/')[1]}-{dataset_id}"

# Define training args
training_args = Seq2SeqTrainingArguments(
    output_dir=repository_id,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=False, # Overflows with fp16
    learning_rate=5e-5,
    num_train_epochs=1,
    # logging & evaluation strategies
    logging_dir=f"{repository_id}/logs",
    logging_strategy="steps",
    logging_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    # metric_for_best_model="overall_f1",
    # push to hub parameters
    report_to="tensorboard",
    push_to_hub=False,
    hub_strategy="every_save",
    hub_model_id=repository_id,
    # hub_token=HfFolder.get_token(), # This line caused the error and is removed
)

# Create Trainer instance
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
# Start training
trainer.train()

Epoch,Training Loss,Validation Loss
1,6.946944,6.475178


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1842, training_loss=7.353254254037731, metrics={'train_runtime': 768.2119, 'train_samples_per_second': 19.176, 'train_steps_per_second': 2.398, 'total_flos': 2738353024794624.0, 'train_loss': 7.353254254037731, 'epoch': 1.0})

In [22]:
from random import randrange

# random sample
sample = dataset['test'][randrange(len(dataset["test"]))]
print("DIALOGUE:\n")
print(sample["dialogue"])

print("\n------------------")
print("REFERENCE SUMMARY:\n")
print(sample["summary"])
text = "summarize: " + sample["dialogue"]

# tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# move tensors to model device
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# generate summary
summary_ids = model.generate(
    **inputs,
    max_new_tokens=128,
    num_beams=4,
    early_stopping=True
)

# decode
summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)



print("\n------------------")
print("MODEL SUMMARY:\n")
print(summary)

DIALOGUE:

Ollie: Okay, Kelly! Ur up nxt!
Kelly: Me? I don't wanna.
Mickey: C'mon!
Jessica: Yeah! What's yours?
Kelly: Fine. It's a sculpture garden in Finnland.
Ollie: What's scary about sculptures? Wait! Do they resemble vampires and stuff?
Mickey: Nah, I'm sure they look rly nice.
Kelly: It's not the sculptures, it's the amount of them and their faces!
Jessica: Faces? What faces?
Kelly: Well, they resemble ppl in different activities like hugging, training, doing sport and so on. But the faces are just morbid and there's like a hundred of them. All staring at you!
Ollie: Another one?
Mickey: Certainly!
Jessica: Well, Ollie, ur turn!
Ollie: Nagoro village in Japan!
Mickey: Y?
Ollie: Well, maybe it's not scary, but it similar to Kelly's place. It's just creepy as hell.
Jessica: Bt y?
Ollie: Imagine a village with ppl living in it. And in the same village u have these human-sized figures. And there's more of them than the ppl that actually live there!
Kelly: Creepy AH!
Mickey: WTF?! Y 